In [1]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

Mounted at /content/drive
Mon Nov  2 17:51:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---------------------

In [10]:
# upload sample_traces.py (or corresponding sample traces generation file)
%cd /content
!cp -rL /content/drive/MyDrive/data .
from google.colab import files
files.upload()
!ls /content/data /content/sample_traces.py

/content


Saving sample_traces.py to sample_traces.py
/content/sample_traces.py

/content/data:
aime  data.md  gsm8k  make_splits.py  math  stats.json	Untitled0.ipynb


In [11]:
!pip install -q vllm math-verify

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [12]:
# confirm installations worked
!python -c "import vllm, math_verify, torch; print('ok', vllm.__version__, torch.cuda.is_available())"

ok 0.33.0 True


In [13]:
# generate and label traces (100 MATH and 100 GSM8K training problems)
%cd /content
!python sample_traces.py --input data/math/train.jsonl --n 100 --out math_sample.jsonl
!python sample_traces.py --input data/gsm8k/train.jsonl --n 100 --out gsm8k_sample.jsonl

/content
W1102 18:02:41.201000 4495 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1102 18:02:41.202000 4495 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1102 18:02:41.203000 4495 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=4495) INFO 11-02 18:02:57 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 5120, 'disable_log_stats': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'}
config.json: 100% 679/679 [00:00<00:00, 1.98MB/s]
(MainProcess pid=4495) INFO 11-02 18:03:16 [model.py:744] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=4495) WARNING 11-02 18:03:16 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=4495) INFO 11-02 18:03:16 [model.py:2170] Using max model len 5120
(MainProcess pid=4495) INFO 11-02 18:03:16 [scheduler.py:310] Chunked prefill is e

In [14]:
# save results
!mkdir -p /content/drive/MyDrive/pilot_results
!cp math_sample.jsonl gsm8k_sample.jsonl /content/drive/MyDrive/pilot_results/

In [36]:
# print summary
import pandas as pd

files = {"math": "math_sample.jsonl", "gsm8k": "gsm8k_sample.jsonl"}
rows = pd.concat(pd.read_json(f, lines=True).assign(source=s) for s, f in files.items())
print(rows.groupby("source")["final_correct"].mean())
print("hit the token limit:", (rows["num_tokens"] >= 4096).mean())

exploded = rows.explode("chunks").dropna(subset=["chunks"])
df = pd.json_normalize(exploded["chunks"].tolist())
df["source"] = exploded["source"].values
print("chunks per question:", len(df) / len(rows))
print(df.groupby("source")["correct"].value_counts(dropna=False))

source
gsm8k    0.86
math     0.69
Name: final_correct, dtype: float64
hit the token limit: 0.185
chunks per question: 5.7
source  correct
gsm8k   True       367
        False      168
        NaN          5
math    False      316
        True       266
        NaN         18
Name: count, dtype: int64


In [38]:
# track answer chunk
df["id"] = exploded["id"].values
df[["id", "k", "end_char", "answer", "correct"]].head(10)

,id,k,end_char,answer,correct
0,math-train-3617,0,1561,13 - i,False
1,math-train-3617,1,2566,3 - 4i,False
2,math-train-3617,2,9098,3 - 4i,False
3,math-train-3617,3,11291,51,True
4,math-train-3617,4,11458,51,True
5,math-train-184,0,3088,36,True
6,math-train-184,1,3834,45,False
7,math-train-184,2,4934,36,True
8,math-train-3668,0,147,7,False
9,math-train-3668,1,896,1,False


In [37]:
# show 50 chunks to verify results
pd.set_option("display.max_colwidth", 200)
labeled_chunks = df[df["correct"].notna()]
labeled_chunks.assign(tail=labeled_chunks["text"].str[-150:])[["source", "tail", "answer", "correct"]].sample(50, random_state=0)

,source,tail,answer,correct
945,gsm8k,"goes to school 6 miles, comes back home 7 miles, which is a round trip, but that's just one trip, so over 5 days, he does that 5 times, so 5*13=65.\n\n",65,True
754,gsm8k,"eparate from the 6 he didn't sell, so total sold: 26, total left: 6, total drunk: 5.\n\nSo 26 + 5 + 6 = 37. But he only made 32, so that's impossible.\n\n",37,False
190,math,"9 + 8 + 0 + 1 = 18. Still too high.\n\nSo, that's not going to work.\n\nWait, so 9700 is the only way when thousands digit is 9. So, that's one number.\n\n",9,False
919,gsm8k,"es he rode this week.\n\nLet me do the multiplication: 13 times 5. Hmm, 10 times 5 is 50, and 3 times 5 is 15, so 50 + 15 is 65. So, 13 times 5 is 65.\n\n",65,True
428,math,"Okay, so I have this equation to solve: \(12^3 = \frac{9^2}{3} \cdot 2^{12p}\). I need to find the value of \(p\).",\frac{9^2}{3} \cdot 2^{12p},False
518,math,"Wait, that makes sense. So, the number of unique assignments is 17.\n\n",17,False
564,math,"sides are in the ratio 1:1:√2, so if the hypotenuse is 6√2, then each leg is 6, which is 6√2 divided by √2. So, that also gives 6. So, same result.\n\n",6,False
903,gsm8k,"n of all three. Then, each of them watched 2 movies in addition to their own unique ones. So, the total unique movies would be 7 +12 +15 -2*2 +2=30.\n\n",30,True
982,gsm8k,"\n\nAnother way: Since Fem is 11, in two years, she'll be 13. Matt is currently 44, so in two years, he'll be 46. So, 13 + 46 is 59. That makes sense.\n\n",59,True
274,math,"3, which is 49 - 3ab = 13.\n\nSo, subtracting 49 from both sides: -3ab = 13 - 49, which is -3ab = -36.\n\nDivide both sides by -3: ab = (-36)/(-3) = 12.\n\n",12,True
